# POLSCI 3

## Week 7 Section Activity: Do Students Change AC Transit Ridership?

This activity covers the *p*-value ideas from Week 6 Lecture 2 and Week 7 Lecture 1: skeptics, null hypotheses, and what a *p*-value does and does not tell us.

- It is intentionally harder than the in-class notebooks. Each question tells you *what* to calculate, but you have to come up with the steps. Use your lecture notebooks and the R cheat sheet, and feel free to talk to others or ask me for help.
- After each question, you can run the **check cell** to see if you are right. For multiple choice, save your letter inside the quotes.
- Cells marked **(given)** have simulation code like the code in lecture, and they calculate the *p*-values for you. You don't need to write or understand the code in them.
- This activity is not graded. For the multiple choice you could just try every letter, but this is for your own learning.

In [ ]:
library(estimatr)
library(ggplot2)

ac <- read.csv('ps3_actransit_ridership.csv')
head(ac)

### Codebook

This data is from AC Transit. Each row is **one AC Transit route, in one month, on one day type**. For instance, one row is the 51B on weekdays in January 2023. Another row is the 51B on Saturdays in January 2023. The data covers 2023 to 2025.

- `route`: the bus route
- `type`: kind of route (`Local`, `Supplemental`, `M-Transbay`, `All Nighter`, `Transbay`). The `All Nighter` routes are the overnight buses.
- `year`, `month`: when the ridership was measured
- `day_type`: `"Weekday"`, `"Saturday"`, or `"Sunday"`
- `avg_daily_riders`: the average number of riders on that route per day, in that month, on that day type
- `all_three_days`: 1 if the route has ridership for **all three** day types in that month, 0 if not
- `near_campus`: 1 for the routes that go near campus (51B, 52, 27, 36, 6, F, 800, and 851), 0 for all other routes. (Route 27 only appears in the data starting in August 2025.)
- `period`: roughly when UC Berkeley classes are in session. `"Semester"` is February to April and September to November, `"Summer"` is June to August, and `"Other"` is December, January, and May (finals and breaks).

---
# Part A: Do students change ridership on campus buses?

You have noticed that the buses near campus feel emptier in the summer. A skeptic says that **students make no difference** to how busy the campus routes are. You think the skeptic is wrong.

For Part A, use only the **campus routes** (`near_campus == 1`), only **weekdays**, and only the months labeled `"Semester"` or `"Summer"` (drop `"Other"`). Each row (one route in one month) is one observation.

**Q1.** Estimate how many more average daily riders the campus routes have on weekdays **in the semester than in the summer**. Save the full `difference_in_means()` output (semester minus summer) as `q1.dim`.

In [ ]:
# Step 1: keep only campus routes, weekdays, and the Semester / Summer months
camp.weekdays <- subset(ac, near_campus == 1 & day_type == 'Weekday' & period != 'Other')

# Step 2: compare semester to summer (baseline = Summer) and save the whole output
q1.dim <- difference_in_means(avg_daily_riders ~ period, data = camp.weekdays,
                              condition1 = 'Summer', condition2 = 'Semester')
q1.dim

**Answer:** The estimate is about 1,023 more riders per route-month during the semester, with a standard error of about 352 (t is about 2.9).

In [ ]:
. = ottr::check("tests/q1.R")

**Q2.** Which pair of hypotheses matches the skeptic's view (the **null**) and what you want to show (the **alternative**)?

- **A.** **Null:** campus routes have more weekday riders in the semester than in the summer. **Alternative:** they have the same number of weekday riders in both.
- **B.** **Null:** campus routes have the same number of weekday riders on average in the semester and the summer (students make no difference). **Alternative:** they have a different number of weekday riders in the semester than in the summer.
- **C.** **Null:** campus routes carry no riders in the summer. **Alternative:** campus routes carry riders in the summer.
- **D.** **Null:** the gap you found in Q1 is exactly the true gap. **Alternative:** the gap you found in Q1 is not exactly the true gap.

Save your choice as a letter (text in quotes) in `q2.answer`.

In [ ]:
q2.answer <- "B"

**Answer: B.** The null hypothesis is what the skeptic believes: no difference. The alternative (what you want to show) is that there is a difference.

In [ ]:
. = ottr::check("tests/q2.R")

### (given) What could happen by chance if students make no difference?

If the skeptic is right, it doesn't matter which rows are labeled Summer or Semester, so we can **shuffle** the labels and see how big a gap luck alone produces, like Week 7 Lecture 1. This cell shuffles the labels 10,000 times and saves:

- `shuffled.sem.sum`: the 10,000 shuffled semester-minus-summer gaps
- `actual.sem.sum`: the gap we actually see in the data
- `p.sem.sum`: the *p*-value, the proportion of shuffled gaps that are **at least as large** as the actual gap

How the *p*-value line works: `shuffled.sem.sum >= actual.sem.sum` answers `TRUE` or `FALSE` for each shuffled gap ("is it at least as large as the actual gap?"). R treats `TRUE` as 1 and `FALSE` as 0, so `mean()` of those is the proportion of `TRUE`s.

(This cell makes its own copy of the data, so it works no matter what you named your objects. Also, `difference_in_means()` prints its own *p*-value in the `Pr(>|t|)` column. It's calculated differently than the simulation here, so don't expect them to match exactly.)

In [ ]:
sim.data <- read.csv('ps3_actransit_ridership.csv')
sim.data <- subset(sim.data, near_campus == 1 & day_type == 'Weekday' & period != 'Other')
sim.data$label <- sim.data$period   # the labels we will shuffle

re.randomize <- function(input.data) {
    input.data$label <- sample(input.data$label)
    return(input.data)
}

compute.gap <- function(input.data, group1, group2) {
    mean(input.data$avg_daily_riders[input.data$label == group1]) -
        mean(input.data$avg_daily_riders[input.data$label == group2])
}

set.seed(22)
shuffled.sem.sum <- replicate(10000, compute.gap(re.randomize(sim.data), 'Semester', 'Summer'))
actual.sem.sum <- compute.gap(sim.data, 'Semester', 'Summer')
p.sem.sum <- mean(shuffled.sem.sum >= actual.sem.sum)

p.sem.sum

The red line is the gap we actually see:

In [ ]:
qplot(shuffled.sem.sum, binwidth = 50) + geom_vline(xintercept = actual.sem.sum, color = 'red')

**Q3.** The *p*-value for semester vs. summer (printed above, about 0.004) is below 0.05. Which statement is correct?

- **A.** There is less than a 5% chance that students make no difference.
- **B.** It proves that campus routes lose exactly the number of riders you found in Q1 every summer.
- **C.** If students made no difference, a gap this large would show up in fewer than 5% of shuffles, so by convention we reject the null.
- **D.** The *p*-value is the size of the gap, in riders.

Save your choice as a letter (text in quotes) in `q3.answer`.

In [ ]:
q3.answer <- "C"

**Answer: C.** A *p*-value is the probability of seeing a gap this large if the null were true. It is not the probability that the null is true (A), it doesn't tell us the exact true gap (B), and it is not the size of the gap (D).

In [ ]:
. = ottr::check("tests/q3.R")

**Q4.** Does the small *p*-value prove that students leaving for the summer **causes** lower ridership on campus routes?

- **A.** Yes: a *p*-value below 0.05 always proves a causal effect.
- **B.** Yes: bigger gaps always mean causation.
- **C.** No: a *p*-value can never be below 0.05 with real data.
- **D.** No: the *p*-value only says luck alone would rarely produce a gap this large if summer and semester ridership were the same. Months aren't randomly assigned, and summer differs in other ways too (for example, vacations and fewer people commuting to jobs), so omitted variable bias is still possible.

Save your choice as a letter (text in quotes) in `q4.answer`.

In [ ]:
q4.answer <- "D"

**Answer: D.** A *p*-value speaks to chance, not to what causes a pattern. This is observational data: summer and the semester differ in many ways besides students being in town, so we should be careful about causal claims (omitted variable bias, Week 4).

In [ ]:
. = ottr::check("tests/q4.R")

---
# Part B: Is 0.05 a magic line?

The skeptic says: "Fine, but maybe that was a fluke of putting all the years together. Check each year separately."

### (given) Semester vs. summer in 2023, 2024, and 2025

This cell does the same shuffle test for the **semester-minus-summer** gap on campus-route weekdays, separately for each year (10,000 shuffles each). It saves the actual gaps (`actual.2023`, `actual.2024`, `actual.2025`) and the *p*-values (`p.2023`, `p.2024`, `p.2025`), and prints them. It again makes its own copy of the data.

In [ ]:
re.randomize <- function(input.data) {
    input.data$label <- sample(input.data$label)
    return(input.data)
}

compute.gap <- function(input.data, group1, group2) {
    mean(input.data$avg_daily_riders[input.data$label == group1]) -
        mean(input.data$avg_daily_riders[input.data$label == group2])
}

shuffle.one.year <- function(yr) {
    d <- read.csv('ps3_actransit_ridership.csv')
    d <- subset(d, near_campus == 1 & day_type == 'Weekday' & period != 'Other' & year == yr)
    d$label <- d$period
    shuffled <- replicate(10000, compute.gap(re.randomize(d), 'Semester', 'Summer'))
    actual <- compute.gap(d, 'Semester', 'Summer')
    list(actual = actual, p = mean(shuffled >= actual))
}

set.seed(2023)
res.2023 <- shuffle.one.year(2023)
res.2024 <- shuffle.one.year(2024)
res.2025 <- shuffle.one.year(2025)

actual.2023 <- res.2023$actual
actual.2024 <- res.2024$actual
actual.2025 <- res.2025$actual

p.2023 <- res.2023$p
p.2024 <- res.2024$p
p.2025 <- res.2025$p

# the actual semester-minus-summer gaps (riders per route-month), then the p-values
c(actual.2023, actual.2024, actual.2025)
c(p.2023, p.2024, p.2025)

**Q5.** The gaps were similar in all three years (roughly 950 to 1,100 more riders per route-month in the semester). A writer looks at your printed *p*-values, which are below 0.05 only in 2025, and says: *"Students made a significant difference in 2025, but in 2023 and 2024 there was no difference."* What is the best response?

- **A.** The 2023 and 2024 evidence is not much weaker than in 2025, and 0.05 is an arbitrary line, so saying there was "no difference" in those years overstates the contrast.
- **B.** That's correct: 2023 and 2024 aren't significant, so there was no difference.
- **C.** *p*-values below 0.05 mean the gap in 2025 was exactly the true gap.
- **D.** 2023 and 2024 must be biased, because their *p*-values are higher.

Save your choice as a letter (text in quotes) in `q5.answer`.

In [ ]:
q5.answer <- "A"

**Answer: A.** The gaps are similar in all three years, and the *p*-values are roughly 0.11 (2023), 0.08 (2024), and 0.04 (2025). Evidence comes in degrees, and "significant" is a convention. Jumping from "significant" to "no difference" because some years land just above the line isn't justified. (Splitting by year also leaves fewer observations in each test, which makes each one noisier.)

In [ ]:
. = ottr::check("tests/q5.R")

---
# Part C: What about night buses?

The `All Nighter` routes (including the 800 and 851 near campus) run overnight. You think night buses are busier on **Saturdays than on weekdays**, because people go out on weekend nights. A skeptic says that the day of the week makes no difference for night buses.

For Part C, use only the routes with `type == 'All Nighter'`, only the year **2024**, and only **Saturdays and weekdays** (drop Sundays). Each row (one route in one month) is one observation.

**Q6.** Estimate how many more average daily riders night buses have on **Saturdays than on weekdays**. Save the full `difference_in_means()` output (Saturday minus weekday) as `q6.dim`.

In [ ]:
# Step 1: keep night buses, 2024, Saturdays and weekdays only
night.sat.wk <- subset(ac, type == 'All Nighter' & year == 2024 & day_type != 'Sunday')

# Step 2: compare Saturday to weekday (baseline = Weekday) and save the whole output
q6.dim <- difference_in_means(avg_daily_riders ~ day_type, data = night.sat.wk,
                              condition1 = 'Weekday', condition2 = 'Saturday')
q6.dim

**Answer:** The estimate is about 16 more riders per route-month on Saturdays, with a standard error of about 27 (t is about 0.6).

In [ ]:
. = ottr::check("tests/q6.R")

**Q7.** Which pair of hypotheses matches the skeptic's view (the **null**) and what you want to show (the **alternative**) for night buses?

- **A.** **Null:** night buses have the same number of average daily riders on Saturdays and weekdays. **Alternative:** night buses have a different number of riders on Saturdays than on weekdays.
- **B.** **Null:** night buses have more riders on Saturdays than on weekdays. **Alternative:** night buses have the same number of riders on both.
- **C.** **Null:** night buses carry no riders on weekdays. **Alternative:** night buses carry riders on weekdays.
- **D.** **Null:** the gap you found in Q6 is exactly the true gap. **Alternative:** the gap you found in Q6 is not exactly the true gap.

Save your choice as a letter (text in quotes) in `q7.answer`.

In [ ]:
q7.answer <- "A"

**Answer: A.** Same logic as Part A: the null is the skeptic's "no difference", and the alternative is what you want to show.

In [ ]:
. = ottr::check("tests/q7.R")

### (given) What could happen by chance for night buses?

Same idea as Part A: shuffle the day-type labels 10,000 times. This cell saves `shuffled.night` (the shuffled Saturday-minus-weekday gaps), `actual.night` (the gap we actually see), and `p.night` (the *p*-value, the proportion of shuffled gaps at least as large as the actual gap). It makes its own copy of the data.

In [ ]:
night.data <- read.csv('ps3_actransit_ridership.csv')
night.data <- subset(night.data, type == 'All Nighter' & year == 2024 & day_type != 'Sunday')
night.data$label <- night.data$day_type   # the labels we will shuffle

re.randomize <- function(input.data) {
    input.data$label <- sample(input.data$label)
    return(input.data)
}

compute.gap <- function(input.data, group1, group2) {
    mean(input.data$avg_daily_riders[input.data$label == group1]) -
        mean(input.data$avg_daily_riders[input.data$label == group2])
}

set.seed(33)
shuffled.night <- replicate(10000, compute.gap(re.randomize(night.data), 'Saturday', 'Weekday'))
actual.night <- compute.gap(night.data, 'Saturday', 'Weekday')
p.night <- mean(shuffled.night >= actual.night)

p.night

**Q8.** In Q6, the Saturday-minus-weekday gap is positive (about 16 more riders per route-month on Saturdays), but the *p*-value printed above is large (about 0.28). What is the best conclusion?

- **A.** Saturdays and weekdays are equally busy on night buses.
- **B.** Weekdays are busier than Saturdays, because the *p*-value is large.
- **C.** The estimate must be wrong, because the *p*-value is above 0.05.
- **D.** The result is inconclusive: we can't rule out chance, though our best guess is that Saturdays are a bit busier.

Save your choice as a letter (text in quotes) in `q8.answer`.

In [ ]:
q8.answer <- "D"

**Answer: D.** A non-significant *p*-value doesn't prove there is no difference. The test is just too noisy to tell (there are only six night routes, and they differ a lot in size). The estimate is still our best guess, but we can't convince the skeptic.

In [ ]:
. = ottr::check("tests/q8.R")

---
# Part D: What if there is nothing to find?

The skeptic has one last challenge: *"If you test enough things, you'll always find something significant."* Let's see if the skeptic is right.

Take the **weekday ridership of every AC Transit route in October 2024**. Imagine **20 student groups**, and each one randomly splits the routes into "Group A" and "Group B", then tests whether Group A has more riders than Group B. The labels mean nothing, so **the null hypothesis is true for every group**.

### (given) The 20 groups

This cell runs the 20 groups' shuffle tests (1,000 shuffles each) and saves each group's *p*-value in `placebo.p`. It prints the 20 *p*-values, rounded. Once again it makes its own copy of the data.

The last two lines show how to count how many of the 20 are below 0.05. `placebo.p < 0.05` answers `TRUE` or `FALSE` for each of the 20 *p*-values. R treats `TRUE` as 1 and `FALSE` as 0, so `sum()` of those *counts* the `TRUE`s. (This is just a demo, so there is nothing to answer here.)

In [ ]:
placebo.data <- read.csv('ps3_actransit_ridership.csv')
placebo.data <- subset(placebo.data, year == 2024 & month == 'Oct' & day_type == 'Weekday')
riders <- placebo.data$avg_daily_riders

set.seed(1)
placebo.p <- replicate(20, {
    random.group <- sample(rep(c('A', 'B'), length.out = length(riders)))     # a meaningless random split
    actual <- mean(riders[random.group == 'A']) - mean(riders[random.group == 'B'])
    shuffled <- replicate(1000, {
        g <- sample(random.group)
        mean(riders[g == 'A']) - mean(riders[g == 'B'])
    })
    mean(shuffled >= actual)
})

round(placebo.p, 3)

placebo.p < 0.05        # TRUE / FALSE for each group
sum(placebo.p < 0.05)   # how many groups are below 0.05

**Q9.** The labels were meaningless, yet one of the 20 groups still got a *p*-value below 0.05. Why?

- **A.** The routes in Group A really are busier than the routes in Group B.
- **B.** The shuffle test is broken.
- **C.** A *p*-value below 0.05 happens about 5% of the time (1 in 20) even when the null is true, just by chance.
- **D.** Real data always has a few significant results.

Save your choice as a letter (text in quotes) in `q9.answer`.

In [ ]:
q9.answer <- "C"

**Answer: C.** This is the jelly bean comic: with 20 tests of a true null, about 1 will come out below 0.05 by luck. (The count can vary with the random seed: it is usually between 0 and about 4, and averages about 1.) That is why a significant *p*-value doesn't prove the null is false.

In [ ]:
. = ottr::check("tests/q9.R")

---
## Done!

If you finish early, try the optional `dplyr` notebook.